# Analysis of OCR Results

This notebook loads OCR results generated through `ecpo-ocr-bench`, reruns the analysis and then tries to identify sources of errors. The idea is to then mitigate these errors and verify by rerunning this notebook.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import collections

In [ ]:
from ecpo_ocr_bench import evaluate_ocr_tool
from IPython.display import JSON

### Data Loading and Processing

In [ ]:
# DATAFILE = "./qwen3-vl-4B/qwen3-vl-4B.json"
# DATAFILE = "./qwen3-vl-30B/qwen3-vl-30B.json"
DATAFILE = "./qwen3-vl-235B/qwen3-vl-235B.json"

In [ ]:
with open(DATAFILE, "r") as f:
    data = json.load(f)

In [ ]:
reprocess = lambda i: data[i.stem]["original_ocr_result"]
data = evaluate_ocr_tool(
    reprocess,
    ignore_linebreaks=True,
    ignore_punctuation=True,
    normalize_punctuation=True,
    normalize_modern_chars=True,
)

In [ ]:
filtered_images = []
filtered_data = data

## General error overview

In [ ]:
distances = {
    i: min(d["distance"], len(d["normalized_ground_truth"]))
    / len(d["normalized_ground_truth"])
    for i, d in data.items()
}

In [ ]:
plt.hist(distances.values(), bins=100)
plt.title(f"CER distribution across {len(data)} samples")
plt.show()

General statistics:

In [ ]:
print(
    f"Mean CER: {np.mean(list(distances.values())):.4f} +- {np.std(list(distances.values())):.4f}"
)

In [ ]:
print(f"Total mistakes in the dataset: {sum((d['distance'] for d in data.values()))}")

In [ ]:
print(
    f"Total mistakes in the part of the dataset that is 90% correct: {sum((d['distance'] for i, d in data.items() if distances[i] < 0.1))}"
)

In [ ]:
print(
    f"Total characters in GT: {sum((len(d['normalized_ground_truth']) for d in data.values()))}"
)

Throw out some terrible ones for manual inspection:

## Error source analysis

### LLM Infinite Loop Breakdown

Criterion for LLM Breakdown: The OCR results is larger than the GT by at least a factor of 3.

In [ ]:
breakdown_cases = [
    i
    for i, d in filtered_data.items()
    if d["distance"] > len(d["normalized_ground_truth"]) * 3
]

In [ ]:
breakdown_cases

In [ ]:
filtered_images.extend(breakdown_cases)

In [ ]:
filtered_data = {i: d for i, d in filtered_data.items() if i not in breakdown_cases}

### Line Break Omission

Prompt following is hard, that is why sometimes we observe that the OCR result is missing linebreaks.

In [ ]:
missing_linebreaks = [
    i for i, d in filtered_data.items() if "\n" in d["editops"]["insertions"]
]

In [ ]:
len(missing_linebreaks)

In [ ]:
number_of_missing_linebreaks = [
    filtered_data[i]["editops"]["insertions"].count("\n")
    / filtered_data[i]["normalized_ground_truth"].count("\n")
    for i in missing_linebreaks
]

In [ ]:
plt.hist(number_of_missing_linebreaks, bins=20)
plt.title(f"Missing linebreak ratio")
plt.show()

### Aspect Ratio issues

Some models (e.g. HRCenterNet need square input to work best). We visually identify similar problems by plotting error against aspect ratio.

In [ ]:
ratios = [
    max(d["width"], d["height"]) / min(d["width"], d["height"]) for d in data.values()
]

In [ ]:
plt.scatter(ratios, distances.values())
plt.title("Aspect Ratio vs. Levenshtein Distance")

Watch out for clusters towards the upper right. These might indicate that the model has issue at higher aspect ratios

### Reading Order Mixups

In [ ]:
def has_mixup(d):
    ocr_chars = list(d["normalized_ocr_result"])
    gt_chars = list(d["normalized_ground_truth"])
    for c in ocr_chars:
        if c in gt_chars:
            gt_chars.remove(c)

    # TODO: Play with this threshold, so that it feels sharp
    return len(gt_chars) < d["distance"] * 0.5

In [ ]:
likely_mixed_up = [i for i, d in filtered_data.items() if has_mixup(d)]

In [ ]:
likely_mixed_up

In [ ]:
JSON(filtered_data["250"])

### Hallucinations into blank space

Possible solution: Insert `Transparent image parts do not belong to the text crop. do no interpret them.` into the prompt, solved it in an example for me (of course after detecting the blank area and masking it in the image).


### Frequent character mixups

These might indicate that we need to extend the modernizations list.

In [ ]:
c = collections.Counter()

In [ ]:
for i, d in filtered_data.items():
    for r in d["editops"]["replacements"]:
        c.update((tuple(r),))

In [ ]:
with open("common-mixups.txt", "w") as f:
    for (detected, gt), number in reversed(sorted(c.items(), key=lambda i: i[1])):
        if number > 1:
            f.write(f"{detected} -> {gt} (happened {number} times)\n")

In [ ]:
print(f"Total mistakes that are part of common mixups: {sum(i for _, i in c.items())}")